In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve
)

print("Ready")

Ready


In [2]:
ROOT = Path.cwd()

DATA_FE_ROOT = ROOT / "data" / "feature_engineering"
OUTPUT_ROOT = ROOT / "outputs" / "threshold_recall_all"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [3]:
DATA_FILE_MAP = {
    "logistic_regression": DATA_FE_ROOT / "logistic_regression" / "telco_churn_fe_logistic.csv",
    "gaussian_nb": DATA_FE_ROOT / "naive_bayes" / "telco_churn_fe_nb.csv",
    "decision_tree": DATA_FE_ROOT / "decision_tree" / "telco_churn_fe_tree.csv",
    "random_forest": DATA_FE_ROOT / "random_forest" / "telco_churn_fe_rf.csv"
}

In [4]:
BEST_PARAMS = {
    "logistic_regression": {
        "C": 50,
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": "balanced",
        "max_iter": 3000,
        "random_state": 42
    },
    "gaussian_nb": {
        "var_smoothing": 1e-12
    },
    "decision_tree": {
        "criterion": "entropy",
        "max_depth": 3,
        "class_weight": "balanced",
        "random_state": 42
    },
    "random_forest": {
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_leaf": 4,
        "max_features": "sqrt",
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1
    }
}

In [5]:
def load_dataset(path):
    df = pd.read_csv(path)

    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    X = df.drop(columns=["Churn"])
    y = df["Churn"]

    return X, y

In [6]:
def build_preprocessor(X, algorithm):
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    if algorithm == "logistic_regression":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])

In [7]:
def build_model(algo):
    if algo == "logistic_regression":
        return LogisticRegression(**BEST_PARAMS[algo])

    elif algo == "gaussian_nb":
        return GaussianNB(**BEST_PARAMS[algo])

    elif algo == "decision_tree":
        return DecisionTreeClassifier(**BEST_PARAMS[algo])

    elif algo == "random_forest":
        return RandomForestClassifier(**BEST_PARAMS[algo])

In [8]:
def evaluate(y_true, y_prob, th):
    y_pred = (y_prob >= th).astype(int)

    return {
        "threshold": th,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "cm": confusion_matrix(y_true, y_pred)
    }

In [9]:
def search_threshold(y_true, y_prob):
    thresholds = np.arange(0.01, 0.91, 0.01)

    rows = []

    for th in thresholds:
        res = evaluate(y_true, y_prob, th)
        rows.append(res)

    df = pd.DataFrame(rows)

    # tối ưu recall
    best = df.loc[df["recall"].idxmax()]

    return df, best["threshold"], best

In [10]:
def run_model(algo):
    print("\n==========", algo, "==========")

    X, y = load_dataset(DATA_FILE_MAP[algo])

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
    )

    pre = build_preprocessor(X_train, algo)
    model = build_model(algo)

    pipe = Pipeline([
        ("pre", pre),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    val_prob = pipe.predict_proba(X_val)[:, 1]
    test_prob = pipe.predict_proba(X_test)[:, 1]

    th_df, best_th, best_row = search_threshold(y_val, val_prob)

    print("Best threshold:", best_th)

    test_default = evaluate(y_test, test_prob, 0.5)
    test_best = evaluate(y_test, test_prob, best_th)

    print("\nDefault (0.5):", test_default)
    print("\nOptimized:", test_best)

    return {
        "algo": algo,
        "best_threshold": best_th,
        "default_recall": test_default["recall"],
        "optimized_recall": test_best["recall"],
        "default_f1": test_default["f1"],
        "optimized_f1": test_best["f1"]
    }

In [11]:
results = []

for algo in ["logistic_regression", "gaussian_nb", "decision_tree", "random_forest"]:
    res = run_model(algo)
    results.append(res)

summary = pd.DataFrame(results)
summary


========== logistic_regression ==========


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/feature_engineering/logistic_regression/telco_churn_fe_logistic.csv'

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(summary["algo"], summary["default_recall"], label="Default")
plt.bar(summary["algo"], summary["optimized_recall"], alpha=0.7, label="Optimized")

plt.title("Recall Comparison")
plt.ylabel("Recall")
plt.legend()
plt.show()

In [ ]:
best = summary.loc[summary["optimized_recall"].idxmax()]

print("Model tốt nhất theo Recall:")
print(best)